# Anexo A4. Multimodal avanzado

**Opcional** · **Dificultad: alta** · Complementa a los notebooks 6 y 7

1. El codificador de imagen de SigLIP 2 por dentro
2. CLIP multilingüe frente a SigLIP 2
3. ColPali / ColQwen en el servidor

In [1]:
from rag_lab import CFG, set_seeds, summary

# CFG.FAST_MODE = False  # corpus de control completo (por defecto se lee FAST_MODE de .env)
# CFG.DOC_PATH = CFG.DATA_DIR / "mi_documento.pdf"  # cualquier PDF sirve
set_seeds()
summary()

Documento:   /home/mikel/github/2627/iab4/reto-1/semana2-reorganizada/data/documento.pdf (existe)
FAST_MODE:   True
Dispositivo: cuda
GEN_MODEL:   google/gemini-3.1-flash-lite
JUDGE_MODEL: openai/gpt-5-mini
VLM_MODEL:   google/gemini-3.1-flash-lite
OpenRouter:  clave configurada


Trazas:      jsonl /home/mikel/github/2627/iab4/reto-1/semana2-reorganizada/trazas/trazas.jsonl  (alumno: profesor)


/home/mikel/github/2627/iab4/reto-1/semana2-reorganizada/rag_lab/tracing.py:49: UserWarning: No se pudo conectar con Langfuse ([Errno 111] Connection refused): las trazas se guardan en JSONL.
  warnings.warn(f"No se pudo conectar con Langfuse ({e}): las trazas se guardan en JSONL.")


## 1. El codificador de imagen por dentro

El codificador de SigLIP 2 (parches de 16, entrada de 256 × 256) devuelve un vector por parche y un vector
resumen de la imagen entera. El vector resumen es el que usa el notebook 7 para buscar.

In [2]:
from PIL import Image
from rag_lab import models, pdf

resized = Image.open(pdf.render_pages()[13]).convert("RGB").resize((256, 256))

In [3]:
import torch
from rag_lab.multimodal import ImageEncoder

model, processor = ImageEncoder("siglip")._siglip()
with torch.no_grad():
    inputs = processor(images=[resized], return_tensors="pt").to(model.device)
    inputs["pixel_values"] = inputs["pixel_values"].to(model.dtype)
    output = model.vision_model(**inputs)
print("Vectores por parche:", tuple(output.last_hidden_state.shape), "-> (imagen, parches, dimensión)")
print("Vector único de la imagen:", tuple(output.pooler_output.shape))
del model, processor, output
models.release()

Vectores por parche: (1, 256, 768) -> (imagen, parches, dimensión)
Vector único de la imagen: (1, 768)


## 2. CLIP multilingüe frente a SigLIP 2

In [4]:
import pandas as pd
from rag_lab import evaluation, models, multimodal, results, versions
from rag_lab.indexes import BM25Index, DenseIndex
from rag_lab.retrieval import rrf

questions = evaluation.load_questions()
visuals = [p for p in questions if p["tipo"] == "visual"]

def page_of(uid):
    return int(uid.split("-")[1])

def correct_position(ranking_ids, correct):
    for i, uid in enumerate(ranking_ids, start=1):
        if page_of(uid) in correct:
            return i
    return None

def metrics(rankings):
    rows = [{"recuperadas": [[page_of(u)] for u in r], "correctas": p["paginas"]} for r, p in zip(rankings, visuals)]
    return evaluation.evaluate_retrieval(rows)

print(len(visuals), "preguntas visuales:", [(p["id"], p["paginas"], p.get("elemento")) for p in visuals])

6 preguntas visuales: [('q15', [1], 'diagrama'), ('q16', [12], 'matriz'), ('q17', [13], 'cronograma'), ('q18', [20], 'cronograma'), ('q19', [1], 'diagrama'), ('q20', [12], 'matriz')]


In [5]:
from rag_lab import multimodal

img_units = multimodal.image_units(with_figures=True)
print(len(img_units), "imágenes:", sum(u["id"].startswith("figura") for u in img_units), "figuras y el resto páginas")

def to_page(ranking):
    by_id = {u["id"]: u for u in img_units}
    seen_pages, output = set(), []
    for uid, _ in ranking:
        page = by_id[uid].get("figura_de", uid)
        if page not in seen_pages:
            seen_pages.add(page)
            output.append(page)
    return output

rank_img, comparison = {}, {}
for family in ["clip", "siglip"]:
    search_index = multimodal.ImageIndex(img_units, family)
    rank_img[family] = [to_page(search_index.search(p["pregunta"], 20))[:10] for p in visuals]
    comparison[family] = metrics(rank_img[family])
    print(family, models.gpu_memory())
    del search_index
    models.release()
pd.DataFrame(comparison).T

22 imágenes: 2 figuras y el resto páginas


clip GPU: 0.55 GB en uso, pico 0.97 GB


siglip GPU: 0.72 GB en uso, pico 0.83 GB


,recall@5,recall@10,MRR@10
clip,0.5,1.0,0.393519
siglip,1.0,1.0,0.888889


In [6]:
image_table = pd.DataFrame(comparison).T
best_image = image_table.sort_values(["recall@5", "MRR@10"], ascending=False).index[0]
print("Enfoque de imagen elegido para V7:", best_image)

Enfoque de imagen elegido para V7: siglip


## 3. ColPali / ColQwen en el servidor

ColQwen indexa cada parche de la página (cientos de vectores por página) y compara cada token de la pregunta
con todos ellos. ColQwen2 necesita unos 4,8 GB de GPU solo para él (medido, ver `servidor/colpali/README.md`) y
ColQwen2.5 no cabe en 6 GB, así que se consulta el servicio del servidor (`COLPALI_URL` en `.env`).
Si no está configurado o no responde, la celda lo indica y el notebook sigue.

In [7]:
rank_colpali = None
if multimodal.colpali_available():
    rank_colpali = [[u for u, _ in multimodal.colpali_search(p["pregunta"], 10)] for p in visuals]
    display(pd.Series(metrics(rank_colpali)))
else:
    print("Servicio ColPali no disponible (COLPALI_URL vacío o sin respuesta): se omite este enfoque.")

Servicio ColPali no disponible (COLPALI_URL vacío o sin respuesta): se omite este enfoque.


In [8]:
from rag_lab import models

models.release()

## Qué hemos visto

- Un VLM o un codificador de imagen convierte cada parche en un vector; el vector resumen representa la imagen entera.
- CLIP y SigLIP dan un vector por imagen; ColPali guarda uno por parche y es mucho más fino, pero necesita más GPU.